# AgeLens — 11b Final Release Package Rebuild

## Purpose

Notebook 11 correctly applied D-017 and updated the live project configuration, but it created the ZIP archive before those final governance updates were written. As a result, the first archive can contain:

- Decision Log version 1.9 without D-017;
- a configuration snapshot from before final package completion.

This repair notebook rebuilds the aggregate-only release package from the **current live project state** after D-017.

## Governance behavior

This notebook:

- does not create a new scientific decision;
- does not change analysis results;
- does not rerun statistical models;
- does not modify the Decision Log or project configuration;
- backs up the existing package and ZIP;
- rebuilds the package using the current D-017-approved files;
- verifies the packaged Decision Log, configuration, reports, checks, and privacy exclusions;
- replaces the stale archive only after every rebuild check passes.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re
import shutil
import zipfile

import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
NOTEBOOK_BUILD = "11b-v2-robust-document-parser"
RELEASE_STAMP = "20260722"
EXPECTED_DECISION = "D-017"
EXPECTED_DECISION_LOG_VERSION = "2.0"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

print(f"Notebook build: {NOTEBOOK_BUILD}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


In [ ]:
def find_project_root(
    folder_name: str = PROJECT_FOLDER_NAME,
) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def extract_document_field(
    text: str,
    field: str,
) -> str:
    """
    Read a field from a Markdown document-control table.

    Accepts common variations such as:
      | Version | 2.0 |
      | Version | 2.0|
      | Version | 2.0
      Version: 2.0
      **Version:** 2.0
      ## Version 2.0
    """
    normalized_field = re.sub(
        r"\s+",
        " ",
        field.strip(),
    ).casefold()

    # Primary parser: split Markdown table rows instead of
    # depending on exact whitespace or a final pipe.
    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        if "|" in line:
            parts = [
                part.strip()
                for part in line.strip("|").split("|")
            ]

            if len(parts) >= 2:
                key = re.sub(
                    r"\s+",
                    " ",
                    parts[0],
                ).casefold()

                if key == normalized_field:
                    value = parts[1].strip()

                    if value:
                        return value

    # Fallbacks for non-table document-control styles.
    fallback_patterns = [
        rf"(?im)^\s*\*\*{re.escape(field)}\s*:\*\*\s*(.+?)\s*$",
        rf"(?im)^\s*{re.escape(field)}\s*:\s*(.+?)\s*$",
        rf"(?im)^\s*#+\s*{re.escape(field)}\s+(.+?)\s*$",
    ]

    for pattern in fallback_patterns:
        match = re.search(
            pattern,
            text,
        )

        if match:
            value = match.group(1).strip().strip("|").strip()

            if value:
                return value

    preview = "\n".join(
        text.splitlines()[:40]
    )

    raise RuntimeError(
        f"Could not extract document field: {field}\n"
        "First 40 document lines:\n"
        f"{preview}"
    )


def copy_required(
    source: Path,
    destination_directory: Path,
) -> Path:
    if not source.exists():
        raise FileNotFoundError(
            f"Required release artifact is missing: {source}"
        )

    destination = destination_directory / source.name
    shutil.copy2(source, destination)
    return destination


def copy_optional_unique(
    docs_root: Path,
    filename: str,
    destination_directory: Path,
) -> Path | None:
    matches = [
        path
        for path in docs_root.rglob(filename)
        if path.is_file()
        and "backup" not in {
            part.lower()
            for part in path.parts
        }
    ]

    if not matches:
        return None

    if len(matches) > 1:
        raise RuntimeError(
            f"Expected at most one current {filename}; "
            f"found {len(matches)}: {matches}"
        )

    destination = destination_directory / matches[0].name
    shutil.copy2(matches[0], destination)
    return destination


PROJECT_ROOT = find_project_root()

CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "agelens_config.json"
)
DECISION_LOG_PATH = (
    PROJECT_ROOT / "docs" / "governance" / "Decision_Log.md"
)
EVIDENCE_GAP_PATH = (
    PROJECT_ROOT
    / "docs"
    / "governance"
    / "Evidence_Gap_Register.md"
)
ASSUMPTION_PATH = (
    PROJECT_ROOT
    / "docs"
    / "governance"
    / "Assumption_Register.md"
)
FINAL_REPORT_PATH = (
    PROJECT_ROOT
    / "docs"
    / "results"
    / "AgeLens_V1_Final_Report.md"
)
CANONICAL_REPORT_PATH = (
    PROJECT_ROOT
    / "docs"
    / "methodology"
    / "Canonical_V1_Rebuild_Report.md"
)
MORTALITY_PROTOCOL_PATH = (
    PROJECT_ROOT
    / "docs"
    / "methodology"
    / "Mortality_Analysis_Protocol.md"
)
MORTALITY_REPORT_PATH = (
    PROJECT_ROOT
    / "docs"
    / "methodology"
    / "Mortality_Model_Validation_Report.md"
)

required_live_files = [
    CONFIG_PATH,
    DECISION_LOG_PATH,
    EVIDENCE_GAP_PATH,
    ASSUMPTION_PATH,
    FINAL_REPORT_PATH,
    CANONICAL_REPORT_PATH,
    MORTALITY_PROTOCOL_PATH,
    MORTALITY_REPORT_PATH,
]

missing_live_files = [
    path
    for path in required_live_files
    if not path.exists()
]

if missing_live_files:
    raise FileNotFoundError(
        f"Required post-D-017 project files are missing: "
        f"{missing_live_files}"
    )

CONFIG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
FIGURES_ROOT = PROJECT_ROOT / CONFIG["paths"]["figures"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
SCRIPTS_ROOT = PROJECT_ROOT / "scripts"
NOTEBOOKS_ROOT = PROJECT_ROOT / "notebooks"
DOCS_ROOT = PROJECT_ROOT / "docs"
RELEASE_ROOT = PROJECT_ROOT / "release"

PACKAGE_ROOT = (
    RELEASE_ROOT
    / f"agelens_v1_release_{RELEASE_STAMP}"
)
ARCHIVE_PATH = (
    RELEASE_ROOT
    / f"agelens_v1_release_{RELEASE_STAMP}.zip"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Current package: {PACKAGE_ROOT}")
print(f"Current archive: {ARCHIVE_PATH}")


## 1. Verify the live post-D-017 state

The document-control reader accepts Markdown table rows with or without a trailing pipe and also supports colon-based or heading-based metadata. This prevents harmless formatting differences in `Decision_Log.md` from blocking the package rebuild.


In [ ]:
decision_text = DECISION_LOG_PATH.read_text(
    encoding="utf-8"
)

decision_version = extract_document_field(
    decision_text,
    "Version",
)

if decision_version != EXPECTED_DECISION_LOG_VERSION:
    raise RuntimeError(
        "The live Decision Log is not the expected post-D-017 "
        f"version. Observed: {decision_version}; "
        f"expected: {EXPECTED_DECISION_LOG_VERSION}."
    )

if "### D-017" not in decision_text:
    raise RuntimeError(
        "D-017 is missing from the live Decision Log."
    )

required_true_gates = [
    "canonical_outputs_regenerated",
    "canonical_regression_checks_passed",
    "cross_sectional_v1_results_allowed",
    "mortality_analysis_authorized",
    "mortality_cohort_defined",
    "mortality_models_completed",
    "mortality_model_validation_passed",
    "mortality_results_allowed",
    "final_release_package_complete",
    "final_report_written",
    "aggregate_release_archive_created",
]

release_gates = CONFIG.get(
    "release_gates",
    {},
)

for gate in required_true_gates:
    if not release_gates.get(gate, False):
        raise RuntimeError(
            f"Required post-D-017 gate is not open: {gate}"
        )

if CONFIG.get(
    "governance",
    {},
).get(
    "final_release_decision"
) != EXPECTED_DECISION:
    raise RuntimeError(
        "The live configuration does not record D-017 as the "
        "final release decision."
    )

final_release_config = CONFIG.get(
    "final_release",
    {},
)

if final_release_config.get(
    "decision"
) != EXPECTED_DECISION:
    raise RuntimeError(
        "The live final_release configuration is not governed "
        "by D-017."
    )

if not final_release_config.get(
    "aggregate_only",
    False,
):
    raise RuntimeError(
        "The live configuration does not mark the package as "
        "aggregate-only."
    )

if final_release_config.get(
    "participant_level_data_included",
    True,
):
    raise RuntimeError(
        "The live configuration unexpectedly allows "
        "participant-level data."
    )

if CONFIG.get(
    "release_scope",
    {},
).get(
    "cause_specific_mortality"
) != "not_authorized":
    raise RuntimeError(
        "Cause-specific mortality is unexpectedly authorized."
    )

print("✅ Live D-017 project state verified.")


## 2. Back up the stale package and archive

In [ ]:
if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Existing release package not found: {PACKAGE_ROOT}"
    )

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        f"Existing release archive not found: {ARCHIVE_PATH}"
    )

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

BACKUP_ROOT = (
    LOGS_ROOT
    / "final_release_package_rebuild_backups"
    / timestamp
)
BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

PACKAGE_BACKUP = (
    BACKUP_ROOT / PACKAGE_ROOT.name
)
ARCHIVE_BACKUP = (
    BACKUP_ROOT / ARCHIVE_PATH.name
)

shutil.copytree(
    PACKAGE_ROOT,
    PACKAGE_BACKUP,
)
shutil.copy2(
    ARCHIVE_PATH,
    ARCHIVE_BACKUP,
)

if sha256_file(ARCHIVE_PATH) != sha256_file(
    ARCHIVE_BACKUP
):
    raise RuntimeError(
        "Release archive backup hash mismatch."
    )

print(f"Package backup: {PACKAGE_BACKUP}")
print(f"Archive backup: {ARCHIVE_BACKUP}")


## 3. Build a temporary package from the current project state

In [ ]:
TEMP_PACKAGE_ROOT = (
    RELEASE_ROOT
    / (
        f".agelens_v1_release_{RELEASE_STAMP}"
        f"_rebuild_{timestamp}"
    )
)
TEMP_ARCHIVE_PATH = (
    RELEASE_ROOT
    / (
        f".agelens_v1_release_{RELEASE_STAMP}"
        f"_rebuild_{timestamp}.zip"
    )
)

if TEMP_PACKAGE_ROOT.exists():
    raise RuntimeError(
        f"Temporary package already exists: {TEMP_PACKAGE_ROOT}"
    )

if TEMP_ARCHIVE_PATH.exists():
    raise RuntimeError(
        f"Temporary archive already exists: {TEMP_ARCHIVE_PATH}"
    )

package_directories = {
    "tables": TEMP_PACKAGE_ROOT / "tables",
    "figures": TEMP_PACKAGE_ROOT / "figures",
    "docs": TEMP_PACKAGE_ROOT / "docs",
    "governance": TEMP_PACKAGE_ROOT / "governance",
    "config": TEMP_PACKAGE_ROOT / "config",
    "scripts": TEMP_PACKAGE_ROOT / "scripts",
    "notebook_inventory": (
        TEMP_PACKAGE_ROOT / "notebook_inventory"
    ),
}

for path in package_directories.values():
    path.mkdir(
        parents=True,
        exist_ok=False,
    )

required_table_names = [
    "08_canonical_regression_checks.csv",
    "10_mortality_ph_diagnostics.csv",
    "10_mortality_reconciliation.csv",
    "10_mortality_release_checks.csv",
    "11_final_cross_sectional_results.csv",
    "11_final_mortality_results.csv",
    "11_release_summary.csv",
    "11_final_release_checks.csv",
]

copied_paths = []

for filename in required_table_names:
    copied_paths.append(
        copy_required(
            TABLES_ROOT / filename,
            package_directories["tables"],
        )
    )

required_figure_names = [
    "11_cross_sectional_cycle_summary.png",
    "11_mortality_forest_plot.png",
]

for filename in required_figure_names:
    copied_paths.append(
        copy_required(
            FIGURES_ROOT / filename,
            package_directories["figures"],
        )
    )

for source in [
    FINAL_REPORT_PATH,
    CANONICAL_REPORT_PATH,
    MORTALITY_PROTOCOL_PATH,
    MORTALITY_REPORT_PATH,
]:
    copied_paths.append(
        copy_required(
            source,
            package_directories["docs"],
        )
    )

for source in [
    DECISION_LOG_PATH,
    EVIDENCE_GAP_PATH,
    ASSUMPTION_PATH,
]:
    copied_paths.append(
        copy_required(
            source,
            package_directories["governance"],
        )
    )

optional_document_names = [
    "00_Research_Protocol_v1.0.md",
    "Research_Protocol.md",
    "Replication_Protocol.md",
    "Validation_Protocol.md",
    "NHANES_Harmonization_Report.md",
    "Methodology.md",
]

optional_copied = []

for filename in optional_document_names:
    copied = copy_optional_unique(
        DOCS_ROOT,
        filename,
        package_directories["docs"],
    )

    if copied is not None:
        optional_copied.append(copied)

copied_paths.append(
    copy_required(
        CONFIG_PATH,
        package_directories["config"],
    )
)

script_paths = sorted(
    path
    for path in SCRIPTS_ROOT.glob("*")
    if path.is_file()
    and path.suffix.lower() in {
        ".r",
        ".py",
        ".sh",
        ".ps1",
    }
)

for source in script_paths:
    destination = (
        package_directories["scripts"]
        / source.name
    )
    shutil.copy2(
        source,
        destination,
    )
    copied_paths.append(destination)

notebook_inventory = pd.DataFrame(
    [
        {
            "notebook": path.name,
            "relative_path": str(
                path.relative_to(PROJECT_ROOT)
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(
            NOTEBOOKS_ROOT.glob("*.ipynb")
        )
        if path.is_file()
    ]
)

NOTEBOOK_INVENTORY_PATH = (
    package_directories["notebook_inventory"]
    / "notebook_inventory.csv"
)
notebook_inventory.to_csv(
    NOTEBOOK_INVENTORY_PATH,
    index=False,
)

package_metadata = {
    "release": "AgeLens V1",
    "release_date": "2026-07-22",
    "rebuild_created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "rebuild_notebook": (
        "11b_release_package_rebuild.ipynb"
    ),
    "governing_decision": EXPECTED_DECISION,
    "decision_log_version": (
        EXPECTED_DECISION_LOG_VERSION
    ),
    "aggregate_only": True,
    "participant_level_data_included": False,
    "cause_specific_mortality_authorized": False,
    "source_archive_backup": str(
        ARCHIVE_BACKUP.relative_to(
            PROJECT_ROOT
        )
    ),
}

PACKAGE_METADATA_PATH = (
    TEMP_PACKAGE_ROOT
    / "release_package_metadata.json"
)
PACKAGE_METADATA_PATH.write_text(
    json.dumps(
        package_metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

readme = """# AgeLens V1 Governed Release Package

This archive was rebuilt after D-017 so that the packaged governance documents and configuration reflect the final live project state.

The package contains aggregate scientific results, figures, governed documentation, configuration, reproducibility scripts, notebook inventory, and file checksums.

It does not contain raw, interim, or participant-level data.

Cause-specific mortality remains unauthorized.
"""

README_PATH = TEMP_PACKAGE_ROOT / "README.md"
README_PATH.write_text(
    readme,
    encoding="utf-8",
)

manifest_candidates = sorted(
    path
    for path in TEMP_PACKAGE_ROOT.rglob("*")
    if path.is_file()
    and path.name != "release_manifest.csv"
)

manifest = pd.DataFrame(
    [
        {
            "path": str(
                path.relative_to(
                    TEMP_PACKAGE_ROOT
                )
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in manifest_candidates
    ]
)

MANIFEST_PATH = (
    TEMP_PACKAGE_ROOT
    / "release_manifest.csv"
)
manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)

print(
    f"Temporary package files: "
    f"{sum(1 for path in TEMP_PACKAGE_ROOT.rglob('*') if path.is_file())}"
)
print(
    f"Optional methodology documents included: "
    f"{[path.name for path in optional_copied]}"
)


## 4. Validate the temporary package before replacement

In [ ]:
PACKAGED_DECISION_LOG = (
    TEMP_PACKAGE_ROOT
    / "governance"
    / "Decision_Log.md"
)
PACKAGED_CONFIG = (
    TEMP_PACKAGE_ROOT
    / "config"
    / "agelens_config.json"
)

packaged_decision_text = (
    PACKAGED_DECISION_LOG.read_text(
        encoding="utf-8"
    )
)
packaged_config = json.loads(
    PACKAGED_CONFIG.read_text(
        encoding="utf-8"
    )
)

package_files = [
    path
    for path in TEMP_PACKAGE_ROOT.rglob("*")
    if path.is_file()
]

forbidden_suffixes = {
    ".parquet",
    ".xpt",
    ".dat",
    ".sas7bdat",
    ".feather",
}

forbidden_files = [
    str(
        path.relative_to(
            TEMP_PACKAGE_ROOT
        )
    )
    for path in package_files
    if path.suffix.lower() in forbidden_suffixes
]

forbidden_path_tokens = {
    "raw",
    "interim",
    "participant",
    "cohort_authorized",
    "complete_case.parquet",
}

suspicious_paths = [
    str(
        path.relative_to(
            TEMP_PACKAGE_ROOT
        )
    )
    for path in package_files
    if any(
        token in str(
            path.relative_to(
                TEMP_PACKAGE_ROOT
            )
        ).lower()
        for token in forbidden_path_tokens
    )
]

package_checks = pd.DataFrame(
    [
        {
            "check": "Packaged Decision Log is version 2.0",
            "pass": (
                extract_document_field(
                    packaged_decision_text,
                    "Version",
                )
                == EXPECTED_DECISION_LOG_VERSION
            ),
            "observed": extract_document_field(
                packaged_decision_text,
                "Version",
            ),
        },
        {
            "check": "Packaged Decision Log contains D-017",
            "pass": (
                "### D-017"
                in packaged_decision_text
            ),
            "observed": (
                "### D-017"
                in packaged_decision_text
            ),
        },
        {
            "check": "Packaged config records D-017",
            "pass": (
                packaged_config[
                    "governance"
                ][
                    "final_release_decision"
                ]
                == EXPECTED_DECISION
            ),
            "observed": packaged_config[
                "governance"
            ].get(
                "final_release_decision"
            ),
        },
        {
            "check": "Packaged config marks final package complete",
            "pass": bool(
                packaged_config[
                    "release_gates"
                ][
                    "final_release_package_complete"
                ]
            ),
            "observed": packaged_config[
                "release_gates"
            ].get(
                "final_release_package_complete"
            ),
        },
        {
            "check": "Packaged config marks aggregate-only release",
            "pass": bool(
                packaged_config[
                    "final_release"
                ][
                    "aggregate_only"
                ]
            ),
            "observed": packaged_config[
                "final_release"
            ].get(
                "aggregate_only"
            ),
        },
        {
            "check": "Packaged config excludes participant-level data",
            "pass": not bool(
                packaged_config[
                    "final_release"
                ][
                    "participant_level_data_included"
                ]
            ),
            "observed": packaged_config[
                "final_release"
            ].get(
                "participant_level_data_included"
            ),
        },
        {
            "check": "No forbidden data-file suffix is included",
            "pass": len(forbidden_files) == 0,
            "observed": forbidden_files,
        },
        {
            "check": "No suspicious participant/raw/interim path is included",
            "pass": len(suspicious_paths) == 0,
            "observed": suspicious_paths,
        },
        {
            "check": "Final report is included",
            "pass": (
                TEMP_PACKAGE_ROOT
                / "docs"
                / "AgeLens_V1_Final_Report.md"
            ).exists(),
            "observed": True,
        },
        {
            "check": "Release manifest is included",
            "pass": MANIFEST_PATH.exists(),
            "observed": MANIFEST_PATH.exists(),
        },
        {
            "check": "Cause-specific mortality remains unauthorized",
            "pass": (
                packaged_config[
                    "release_scope"
                ][
                    "cause_specific_mortality"
                ]
                == "not_authorized"
            ),
            "observed": packaged_config[
                "release_scope"
            ].get(
                "cause_specific_mortality"
            ),
        },
    ]
)

if not package_checks["pass"].all():
    display(
        package_checks.loc[
            ~package_checks["pass"]
        ]
    )
    raise RuntimeError(
        "Temporary post-D-017 release package failed "
        "validation. The existing release was not replaced."
    )

display(package_checks)

print("✅ Temporary post-D-017 package validated.")


## 5. Create and verify the temporary ZIP archive

In [ ]:
with zipfile.ZipFile(
    TEMP_ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        file
        for file in TEMP_PACKAGE_ROOT.rglob("*")
        if file.is_file()
    ):
        archive.write(
            path,
            arcname=(
                f"{PACKAGE_ROOT.name}/"
                + str(
                    path.relative_to(
                        TEMP_PACKAGE_ROOT
                    )
                ).replace("\\", "/")
            ),
        )

if not TEMP_ARCHIVE_PATH.exists():
    raise RuntimeError(
        "Temporary release ZIP was not created."
    )

with zipfile.ZipFile(
    TEMP_ARCHIVE_PATH,
    mode="r",
) as archive:
    bad_entry = archive.testzip()

    if bad_entry is not None:
        raise RuntimeError(
            f"ZIP integrity failure at: {bad_entry}"
        )

    zip_names = set(
        archive.namelist()
    )

    required_zip_entries = {
        (
            f"{PACKAGE_ROOT.name}/"
            "governance/Decision_Log.md"
        ),
        (
            f"{PACKAGE_ROOT.name}/"
            "config/agelens_config.json"
        ),
        (
            f"{PACKAGE_ROOT.name}/"
            "docs/AgeLens_V1_Final_Report.md"
        ),
        (
            f"{PACKAGE_ROOT.name}/"
            "release_manifest.csv"
        ),
    }

    missing_zip_entries = sorted(
        required_zip_entries - zip_names
    )

    if missing_zip_entries:
        raise RuntimeError(
            "Temporary ZIP is missing required entries: "
            f"{missing_zip_entries}"
        )

    zipped_decision_text = archive.read(
        (
            f"{PACKAGE_ROOT.name}/"
            "governance/Decision_Log.md"
        )
    ).decode("utf-8")

    zipped_config = json.loads(
        archive.read(
            (
                f"{PACKAGE_ROOT.name}/"
                "config/agelens_config.json"
            )
        ).decode("utf-8")
    )

if "### D-017" not in zipped_decision_text:
    raise RuntimeError(
        "The ZIP copy of the Decision Log does not contain D-017."
    )

if extract_document_field(
    zipped_decision_text,
    "Version",
) != EXPECTED_DECISION_LOG_VERSION:
    raise RuntimeError(
        "The ZIP copy of the Decision Log is not version 2.0."
    )

if zipped_config[
    "governance"
].get(
    "final_release_decision"
) != EXPECTED_DECISION:
    raise RuntimeError(
        "The ZIP configuration does not record D-017."
    )

new_archive_sha256 = sha256_file(
    TEMP_ARCHIVE_PATH
)

print("✅ Temporary ZIP integrity and governance verified.")
print(f"Temporary ZIP SHA-256: {new_archive_sha256}")


## 6. Replace the stale package atomically and write the repair audit

In [ ]:
old_archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

shutil.rmtree(
    PACKAGE_ROOT
)
TEMP_PACKAGE_ROOT.rename(
    PACKAGE_ROOT
)

ARCHIVE_PATH.unlink()
TEMP_ARCHIVE_PATH.rename(
    ARCHIVE_PATH
)

final_archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if final_archive_sha256 != new_archive_sha256:
    raise RuntimeError(
        "Final archive hash differs from the validated "
        "temporary archive."
    )

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="r",
) as archive:
    final_decision_text = archive.read(
        (
            f"{PACKAGE_ROOT.name}/"
            "governance/Decision_Log.md"
        )
    ).decode("utf-8")

    final_config = json.loads(
        archive.read(
            (
                f"{PACKAGE_ROOT.name}/"
                "config/agelens_config.json"
            )
        ).decode("utf-8")
    )

final_checks = pd.DataFrame(
    [
        {
            "check": "Canonical release directory exists",
            "pass": PACKAGE_ROOT.exists(),
            "observed": str(PACKAGE_ROOT),
        },
        {
            "check": "Canonical release ZIP exists",
            "pass": ARCHIVE_PATH.exists(),
            "observed": str(ARCHIVE_PATH),
        },
        {
            "check": "Final ZIP Decision Log is version 2.0",
            "pass": (
                extract_document_field(
                    final_decision_text,
                    "Version",
                )
                == EXPECTED_DECISION_LOG_VERSION
            ),
            "observed": extract_document_field(
                final_decision_text,
                "Version",
            ),
        },
        {
            "check": "Final ZIP contains D-017",
            "pass": (
                "### D-017"
                in final_decision_text
            ),
            "observed": (
                "### D-017"
                in final_decision_text
            ),
        },
        {
            "check": "Final ZIP config records D-017",
            "pass": (
                final_config[
                    "governance"
                ][
                    "final_release_decision"
                ]
                == EXPECTED_DECISION
            ),
            "observed": final_config[
                "governance"
            ].get(
                "final_release_decision"
            ),
        },
        {
            "check": "Archive hash changed from stale package",
            "pass": (
                final_archive_sha256
                != old_archive_sha256
            ),
            "observed": {
                "old": old_archive_sha256,
                "new": final_archive_sha256,
            },
        },
        {
            "check": "Stale package backup exists",
            "pass": PACKAGE_BACKUP.exists(),
            "observed": str(PACKAGE_BACKUP),
        },
        {
            "check": "Stale archive backup exists",
            "pass": ARCHIVE_BACKUP.exists(),
            "observed": str(ARCHIVE_BACKUP),
        },
    ]
)

if not final_checks["pass"].all():
    display(
        final_checks.loc[
            ~final_checks["pass"]
        ]
    )
    raise RuntimeError(
        "Post-replacement release-package verification failed. "
        f"Backups remain at {BACKUP_ROOT}."
    )

CHECKS_PATH = (
    TABLES_ROOT
    / "11b_release_package_rebuild_checks.csv"
)
MANIFEST_AUDIT_PATH = (
    TABLES_ROOT
    / "11b_release_package_manifest.csv"
)
METADATA_PATH = (
    LOGS_ROOT
    / "11b_release_package_rebuild_metadata.json"
)

package_checks.to_csv(
    CHECKS_PATH,
    index=False,
)

final_manifest = pd.read_csv(
    PACKAGE_ROOT / "release_manifest.csv"
)
final_manifest.to_csv(
    MANIFEST_AUDIT_PATH,
    index=False,
)

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "11b_release_package_rebuild.ipynb"
    ),
    "notebook_build": NOTEBOOK_BUILD,
    "governing_decision": EXPECTED_DECISION,
    "decision_log_version": (
        EXPECTED_DECISION_LOG_VERSION
    ),
    "old_archive_sha256": old_archive_sha256,
    "new_archive_sha256": final_archive_sha256,
    "package_directory": str(
        PACKAGE_ROOT.relative_to(
            PROJECT_ROOT
        )
    ),
    "archive": str(
        ARCHIVE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "package_file_count": int(
        sum(
            1
            for path in PACKAGE_ROOT.rglob("*")
            if path.is_file()
        )
    ),
    "participant_level_data_included": False,
    "cause_specific_mortality_authorized": False,
    "backup_root": str(
        BACKUP_ROOT.relative_to(
            PROJECT_ROOT
        )
    ),
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

display(final_checks)

print("✅ Post-D-017 release package rebuild completed.")
print("✅ Decision Log v2.0 and D-017 are inside the ZIP.")
print("✅ Final configuration snapshot is inside the ZIP.")
print("✅ Aggregate-only privacy checks passed.")
print("No scientific results or governance decisions changed.")
print(f"Final archive: {ARCHIVE_PATH}")
print(f"New archive SHA-256: {final_archive_sha256}")
print(f"Stale-package backup: {BACKUP_ROOT}")
